#### 연습 문제
1. data 폴더 안에 rating_train.csv 파일을 로드
2. 결측치를 제외
3. id 컬럼 제외
4. 중복 데이터 제거
4. train 데이터를 생성하기 위해서 label이 0인 데이터중 2000개를 추출
5. label이 1인 데이터중 2000개를 추출
6. 5번 6번의 결과를 단순 행결합
7. train, test 데이터셋을 8:2의 비율로 나눠준다.
8. tokenizer는 Okt를 사용
9. 불필요한 품사를 제외 (사용할 품사 : 명사, 동사, 형용사, 부사, 파티클)
10. 글자 수의 제한은 2자리부터 가능
11. tfidf를 사용하여 벡터화
    - min_df = 3
    - ngram_range = (1, 2)
12. 로지스틱회귀 모델을 사용하여 벡터화한 데이터에서 학습, random_state만 42로 고정
13. test 데이터셋을 이용하여 검증후 평가지표
14. 예측 결과, 원본의 데이터셋과 예측 확률을 하나의 데이터프레임으로 생성
15. 결과물 제출 -> 평가지표, 14번의 결과에서 상위 5개

In [2]:
import pandas as pd
from konlpy.tag import Okt
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [5]:
import numpy as np
df['document'] = df['document'].str.strip()
df.dropna(inplace = True)


In [6]:
df.drop_duplicates('document', inplace=True)
df.drop('id', axis=1, inplace=True)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146182 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  146182 non-null  object
 1   label     146182 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.3+ MB


In [8]:
df_0 = df[df['label'] == 0].sample(n=2000, random_state=42)
df_1 = df[df['label'] == 1].sample(n=2000, random_state=42)

In [9]:
df = pd.concat([df_0, df_1], axis = 0).reset_index(drop = True)

In [10]:
x = df['document'].values
y = df['label'].values

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state=42, stratify = y
)

In [12]:
okt = Okt()

In [13]:
def my_tokenizer(text):
    allow_pos = ['Noun', 'Verb', 'Adjective', 'Adverb', 'KoreanParticle']
    
    # 형태소 분석 및 품사 태깅
    tokens = okt.pos(text, stem=True) # stem=True를 통해 어간 추출 (예: '먹었다' -> '먹다')
    
    # 10. 글자 수 2자리 이상 + 지정된 품사 필터링
    # 이전에 질문하신 '무단', '전재' 등 불용어 제거를 여기에 추가할 수도 있습니다.
    result = [word for word, pos in tokens if pos in allow_pos and len(word) >= 2]
    return result

In [14]:
tfidf_vec = TfidfVectorizer(
    tokenizer = my_tokenizer,
    ngram_range = (1, 2),
    min_df = 2
)

In [15]:
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

c:\Python310\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [16]:
# 생성된 피처의 개수
print(len(tfidf_vec.get_feature_names_out()))

3905


In [17]:
model = LogisticRegression(random_state = 42)
model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [18]:
pred = model.predict(X_test_tfidf)

In [19]:
prob = model.predict_proba(X_test_tfidf)

In [20]:
pd.DataFrame(list(zip(
    X_test, pred, prob
)))

,0,1,2
0,최고의 sf 영화... 말이 필요없다,1,"[0.08639065283036151, 0.9136093471696385]"
1,귀엽고재밌었어요!!! 2005년에는 내용이 아직 받아들이기 힘들었나봐용,1,"[0.24891868621129076, 0.7510813137887092]"
2,99분의 시간이 짧음에도 불구하고 지루함이 느껴진다..,0,"[0.6751904451576521, 0.32480955484234786]"
3,사스가...노무라 만사이 최고의영화,1,"[0.10667823404965138, 0.8933217659503486]"
4,"다시보고싶은영화,여러군데 찾아다녀도 없네ㅠ.ㅠ",1,"[0.415077135397084, 0.584922864602916]"
...,...,...,...
795,명작 입니다 꼭보셔야합니다 꼭보세요 꼭 돈내고보세요,1,"[0.21822844963631882, 0.7817715503636812]"
796,이런게 왜 9점대야...?,0,"[0.6091796410650361, 0.39082035893496386]"
797,나쁘진않네요,1,"[0.4874505841818927, 0.5125494158181073]"
798,"주윤발의 연기력은 볼만하나 지금 보기엔 스토리나 연출력이 영,.~아냐",0,"[0.7790220633324592, 0.22097793666754081]"


In [21]:
# X_test, pred, proba 3개의 데이터를 반복문을 통해서 반복 실행 -> 2차원으로 새로운 데이터를 구성
data = []
for review, value, prob in zip(X_test, pred, prob):
    # value는 1이라면 '긍정', 0이라면 '부정'
    value = '긍정' if value == 1 else '부정'
    # prob -> 둘중에 큰 값만 사용 -> 100을 곱한다. -> 소수점 3번째 자리에서 반올림 -> '%' 붙여준다.
    prob = round(max(prob) * 100, 2)
    prob = str(prob) + '%'
    # review, value, prob 데이터를 하나의 리스트에 대입
    data.append([review, value, prob])
data

[['최고의 sf 영화... 말이 필요없다', '긍정', '91.36%'],
 ['귀엽고재밌었어요!!! 2005년에는 내용이 아직 받아들이기 힘들었나봐용', '긍정', '75.11%'],
 ['99분의 시간이 짧음에도 불구하고 지루함이 느껴진다..', '부정', '67.52%'],
 ['사스가...노무라 만사이 최고의영화', '긍정', '89.33%'],
 ['다시보고싶은영화,여러군데 찾아다녀도 없네ㅠ.ㅠ', '긍정', '58.49%'],
 ['좋너', '긍정', '88.27%'],
 ['진짜 ㅋㅋㅋㅋ개노답 아오빡쳐', '부정', '51.7%'],
 ['부인의불륜을 남편이 정리한 영화인것같군요.그이상 모가있을까요?', '부정', '50.78%'],
 ['감사합니다 덕분에 안보고 잡니다.', '긍정', '58.81%'],
 ['23세 여자임ㅋㅋ영 아이 영화인줄알았는데 완전꿀잼♥ 평점 왤캐낮은지 이해불가...ㅋㅋ캐릭터도넘귀엽고 웃겨서 좋았음 이런영화 너무좋다ㅠㅠ',
  '긍정',
  '85.29%'],
 ['여자들이 보기에나 좋은 영화인듯 동물 좋아해서 봤는데 중반부터 여자주인공 시작부터 끝까지 짜증내는것만나오네',
  '부정',
  '53.47%'],
 ['이런 형편없는 영화에 3500원 지출하여 30분 감상에 눈물을 머금고 포기.', '부정', '58.73%'],
 ['광신도들에의해과대평과받은영화라고나 할까? 참고로 모태신앙인.', '부정', '67.23%'],
 ['다른직업 찾아보길 권장합니다', '긍정', '50.5%'],
 ['웃기기는커녕 유치하구 시답잖네.', '부정', '73.51%'],
 ['이게 마블식 헐리웃 액션인가', '긍정', '52.63%'],
 ['영화는 좋은데 성우는 좋지않는다... 일본 성우 목소리 들었다면 완전한 대박을 날것 같았다..내가 보는 애니중에 제일 감동있고 뭔가 신비롭고 표현할수 없는 무엇과가 있다 이 < 천년여우 여우비 >를 보고 난후에..',
  '긍정',
  '66.52%'],
 ['0점조 아깝고 마니너스 점수를 준다

In [22]:
pd.DataFrame(data, columns = ['review', 'Pred', 'Prob']).head()

,review,Pred,Prob
0,최고의 sf 영화... 말이 필요없다,긍정,91.36%
1,귀엽고재밌었어요!!! 2005년에는 내용이 아직 받아들이기 힘들었나봐용,긍정,75.11%
2,99분의 시간이 짧음에도 불구하고 지루함이 느껴진다..,부정,67.52%
3,사스가...노무라 만사이 최고의영화,긍정,89.33%
4,"다시보고싶은영화,여러군데 찾아다녀도 없네ㅠ.ㅠ",긍정,58.49%


In [23]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.75      0.77      0.76       400
           1       0.77      0.74      0.75       400

    accuracy                           0.76       800
   macro avg       0.76      0.76      0.76       800
weighted avg       0.76      0.76      0.76       800



- 네이버 개발자센터를 이용해서 데이터를 수집
- 수집된 데이터를 이용하여 감정 평가 예측
    1. 네이버 개발자센터 접속
    2. 서비스 api 신청
    3. 서비스키를 이용해서 뉴스 데이터를 로드
    4. 데이터를 이용하여 감정분석

In [37]:
import os
from dotenv import load_dotenv
import requests
import re

In [38]:
load_dotenv()

True

In [39]:
naver_id = os.getenv('naver_api_id')
naver_secret = os.getenv('naver_api_secret')

In [40]:
naver_id

'iEbfZAMk4A7Fk60VGAmB'

In [41]:
# 네이버 api를 활용해서 news 제목들을 수집
url = "https://openapi.naver.com/v1/search/news.json"

params = {
    'query' : '왕사남',
    'display' : 30
}

headers = {
    'X-Naver-Client-Id' : naver_id,
    'X-Naver-Client-Secret' : naver_secret
}

res = requests.get(
    url,
    params = params,
    headers = headers
)
res

<Response [200]>

1. res.json()에서 title 부분의 value를 추출하여 하나의 리스트로 생성
2. <b>, </b> 문자를 제거
3. 위에서 만들어둔 벡터화를 이용하여 벡터화 작업
4. 로지스틱 모델을 이용하여 예측
5. 예측값과 확률을 데이터 프레임으로 생성

In [42]:
new_titles = []
for item in res.json()['items']:
    # item -> dict 형태 데이터가 대입
    # print(itme['title'].replace('<b>', '').replace('</b>', ''))
    # break
    clean_title = re.sub(r'<[^>]*>', '', item['title'])
    # print(clean_title)
    new_titles.append(clean_title)

In [43]:
X_api = tfidf_vec.transform(new_titles)

In [44]:
X_api.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(30, 3905))

In [45]:
pred_api = model.predict(X_api)
proba_api = model.predict_proba(X_api)

In [49]:
data = []
for title, value, proba in zip(new_titles, pred_api, proba_api):
    value = '긍정' if value == 1 else '부정'
    proba = round(max(proba)*100, 2)
    data.append({
        'title' : title,
        'pred' : value,
        'pred_proba' : proba
    })
df_api = pd.DataFrame(data)

In [51]:
df_api.sort_values('pred_proba', ascending = False).head(10)

,title,pred,pred_proba
2,‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다,부정,76.54
22,"[더벨][매니저 프로파일 | 쏠레어파트너스] 영화 현장 20년, 시나리오서...",부정,67.70
29,"‘군체’, ‘왕사남’보다 빠르다…10일 만에 300만 손익분기점 돌파",긍정,62.89
5,"밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행",부정,62.64
25,"“‘왕사남’보다 빠르다, 올해 개봉작 중 1등”…‘군체’ 310만 돌파",긍정,60.54
17,"연상호 감독 '군체', 400만 향해 돌진 중-'왕사남' 넘을까?",부정,60.04
26,"‘군체’ 벌써 300만 돌파, ‘왕사남’보다 빠르다 [지금뉴스]",긍정,59.61
16,한강이 연 ‘소설의 시대’ 여전 …상반기 베스트셀러 1~3위 휩쓸어,부정,59.51
7,'왕사남' 열풍에 찻사발축제 효과 '톡톡'…문경새재 방문객 153만 명 돌...,부정,59.33
24,"연간 2위 '군체', 350만명 최단기 기록…'왕사남'보다 2일 빨라",긍정,58.59


#### 모델의 성능을 올려보자
- 실제 모델의 성능
    - 정확도 : 77%
- 모델의 성능을 올릴수 있는 방법
    - 데이터 양을 늘린다
    - 전처리 방법을 다른 방법으로 사용 (분석기, 벡터화)
    - 스케일러를 이용
    - 벡터화 모델과 분류 모델의 파라미터 수정
    - 모델을 변경

In [ ]:
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

In [ ]:
ma_scaler = MaxAbsScaler()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline(
    [
        (
            'vec', tfidf_vec
        ), 
        {
            'scaler', ma_scaler
        }, 
        (
            'model', model
        )
    ]
)

In [ ]:
# 파라미터 조합 생성 
params = {
    'vec__min_df' : [2, 3], 
    'vec__ngram_range' : [ (1, 2), (1, 1) ], 
    'vec__max_features' : [ None, 1000 ], 
    'model__C' : [0.8, 0.9, 1.0], 
}

In [ ]:
grid = GridSearchCV(
    estimator = pipe, 
    param_grid = params, 
    cv = cv, 
    verbose=1
)

In [ ]:
grid.fit(X, y)

In [ ]:
grid.best_params_

In [ ]:
pred_api = grid.predict(new_titles)
proba_api = grid.predict_proba(new_titles)

In [ ]:
data = []
for title, value, proba in zip(new_titles, pred_api, proba_api):
    value = '긍정' if value == 1 else '부정'
    proba = round( max(proba) * 100, 2 )
    data.append( 
        {
            'title' : title, 
            'pred' : value, 
            'pred_proba' : proba
        }
    )

df_api = pd.DataFrame(data)

In [ ]:
df_api.head(10)

In [ ]:
load_dotenv()

In [ ]:
youtube_api = os.getenv('youtube_api')

- 유튜브에서 특정 영상의 댓글을 로드
    - 구글 클라우드 콘솔에서 api를 신청
    - 신청이 된 api key와 영상의 id 값이 필요
    - 라이브러리 설치
    - google-api-python-client 라이브러리 설치

In [ ]:
# !pip install google-api-python-client

   ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
   ------------------------- -------------- 9.7/15.3 MB 50.5 MB/s eta 0:00:01
   ---------------------------------------- 15.3/15.3 MB 43.8 MB/s  0:00:00

   ---- -----------------------------------  1/10 [pyasn1]
   ---- -----------------------------------  1/10 [pyasn1]
   ---- -----------------------------------  1/10 [pyasn1]
   ---- -----------------------------------  1/10 [pyasn1]
   ---- -----------------------------------  1/10 [pyasn1]
   -------- -------------------------------  2/10 [proto-plus]
   -------- -------------------------------  2/10 [proto-plus]
   -------- -------------------------------  2/10 [proto-plus]
   -------- -------------------------------  2/10 [proto-plus]
   ---------------- -----------------------  4/10 [googleapis-common-protos]
   ---------------- -----------------------  4/10 [googleapis-common-protos]
   ---------------- -----------------------  4/10 [googleapis-common-proto


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# !pip install google-api-python-client
# 영상의 id를 하나 복사
# 유튜브 영상 url에서 가장 마지막 v=ID
video_id = 'z8BW9Fo9-wE'

In [ ]:
from googleapiclient.discovery import build

In [ ]:
youtube = build('youtube', 'v3', developerKey=youtube_api)

In [ ]:
request = youtube.commentThreads().list(
    part = 'snippet', 
    videoId = video_id, 
    maxResults = 10, 
    textFormat = 'plainText'
)

In [ ]:
res = request.execute()

In [ ]:
from pprint import pprint

In [ ]:
comment = []
for item in res['items']:
    comment.append(item['snippet']['topLevelComment']['snippet']['textDisplay'])
    # break

In [ ]:
comment

In [ ]:
labels = [0, 0, 0, 0, 1, 1, 1, 0, 0, 1]
commnet_df = pd.DataFrame(zip(comment, labels), columns = ['review', 'label'])
commnet_df

In [ ]:
pred = grid.predict(comment)
pred_proba = grid.predict_proba(comment)

In [ ]:
pred

In [ ]:
print(classification_report(pred, labels))

In [ ]:
data = []
for review, label, value, proba in zip(comment, labels, pred, pred_proba):
    label = '긍정' if label == 1 else '부정'
    value = '긍정' if value == 1 else '부정'
    proba = round( max(proba) * 100 , 2)
    proba = str(proba) + '%'

    data.append(
        {
            'review' : review, 
            'origin' : label, 
            'pred' : value, 
            'proba' : proba
        }
    )

df_youtube = pd.DataFrame(data)
df_youtube